*Module 3 of 9*

> **¿Prefieres español?** Abre [`03_datos_limpios_geomediana.ipynb`](../es/03_datos_limpios_geomediana.ipynb) — es el mismo módulo, en español.


# ☁️ Module 3 — Clean data: from clouds to the geomedian

🧭 **Objectives** — understand why raw satellite images are messy (clouds,
shadows, gaps), what a **cloud mask** and **quality flags** do, and how a
**geomedian** turns a whole month of imperfect images into one clean,
gap-free image — the tile you have been using.

📚 **The problem.** A single satellite pass is often ruined by clouds and
their shadows. One image of your field might be 40% cloud. The fix is
**temporal compositing**: take *many* images over a period and combine them
per pixel, keeping only good observations.

📚 **The geomedian.** A plain per-band median would pick, say, the median red
from one date and the median NIR from another — breaking the pixel's true
color. The **geomedian** (geometric median) instead finds the single
multi-band value closest to all cloud-free observations *at once*, so band
ratios like NDVI stay physically consistent. Clouds are outliers, so they
get voted out. The result is **Analysis Ready Data**: surface reflectance,
cloud-free, ready to use.

![the geomedian](../../anim/en/03_geomedian.svg)


## Where the data comes from (and the optional GEE)

The tile is built from **NASA HLS** (Harmonized Landsat + Sentinel-2),
streamed from open **STAC/COG** catalogs — no account needed. The production
pipeline (`geocrop_analysis_mx`) can *optionally* use Google Earth Engine
too, but it is not required: the same geomedian can be built from open
catalogs. You will meet this choice again in Module 9.


## Simulate the problem, then the fix

You do not have the raw cloudy stack in the browser, but you can *feel* why
the geomedian works with a tiny experiment: take one clean pixel value,
scatter fake "cloudy" observations on top (clouds are bright — high values),
and watch how the **median** ignores them while the **mean** is fooled.


In [ ]:
import numpy as np

# 10 observations of one pixel's red reflectance over a month.
# 7 are clear (~0.06); 3 are cloud-contaminated (bright, ~0.8).
observations = np.array([0.06, 0.05, 0.80, 0.07, 0.06, 0.78, 0.05, 0.82, 0.06, 0.07])

print("Mean   (fooled by clouds):", round(observations.mean(), 3))
print("Median (votes clouds out):", round(np.median(observations), 3))
print("True clear value is about 0.06 — the median recovers it.")

## The geomedian is already in your tile

Every pixel of your tile is the *result* of this process applied across a
month of HLS images, for all bands together. That is why it looks seamless —
no cloud holes, consistent color. Load it and confirm it is complete
(no missing pixels in the valid area).


In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

In [ ]:
import rasterio, matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()

# A geomedian has no cloud gaps: count pixels that are exactly 0 (nodata)
valid = np.count_nonzero(img[3] != 0)         # NIR band
total = img[3].size
print(f"Valid pixels: {valid:,} of {total:,} ({100*valid/total:.1f}%)")

rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
plt.figure(figsize=(7, 7)); plt.imshow(rgb)
plt.title("Clean geomedian — no clouds, no gaps"); plt.axis("off"); plt.show()

## 🧪 Check yourself

**Why not just average all the images of a month to remove clouds?**

<details><summary>Show answer</summary>

Clouds are bright outliers; the **mean** gets dragged upward by them. The
**median** (and the multi-band geomedian) ignores outliers, so cloudy
observations are effectively voted out. Averaging would leave a hazy,
cloud-tinted image.

</details>

**What does the "geo" in geomedian buy you over a per-band median?**

<details><summary>Show answer</summary>

It keeps the bands *consistent per pixel*: instead of mixing the red from
one date with the NIR from another, it picks one multi-band observation
closest to all clear ones at once. That keeps band ratios like NDVI
physically meaningful.

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Cloud masking](https://abxda.github.io/rs-learning-audio/?id=cloud-masking)
- [Quality flags](https://abxda.github.io/rs-learning-audio/?id=quality-flags)
- [Analysis Ready Data (ARD)](https://abxda.github.io/rs-learning-audio/?id=analysis-ready-data)
- [Temporal compositing](https://abxda.github.io/rs-learning-audio/?id=temporal-compositing)
- [Harmonized Landsat-Sentinel (HLS)](https://abxda.github.io/rs-learning-audio/?id=harmonized-landsat-sentinel)
- [Surface reflectance](https://abxda.github.io/rs-learning-audio/?id=surface-reflectance)



---

[← Previous · Module 2 — How a satellite sees the world](02_how_a_satellite_sees.ipynb) · [Next → · Module 4 — Vegetation indices](04_vegetation_indices.ipynb)
